# Notebook 02.6 — Alpha Sweep (Layer 4 & 8)

Runs 30k-step pilots for several alpha values to find the setting
that best balances sparsity, reconstruction, and feature diversity.

**Goal**: reduce mean encoder cosine similarity (redundancy) while keeping dead% ≈ 0.
Target: mean_sim < 0.20, dead < 2%, L0 in range 25–60.

Layer 12 is kept unchanged (already healthy: mean_sim=0.071).

After this notebook, copy the winning alpha values into `02_sae_training.ipynb` and retrain fully.

In [1]:
import sys, gc
sys.path.insert(0, '..')
import torch
import torch.nn.functional as F
from src.sae import SAE

DEVICE        = 'cuda'
D_INPUT       = 768
D_HIDDEN      = 3072
LR            = 1e-4
BATCH         = 2048
PILOT_STEPS   = 30_000
PILOT_WARMUP  = 10_000
PILOT_RESAMPLE_STEP = 20_000
LOG_EVERY     = 2_000

ALPHAS_L4 = [4e-3, 6e-3, 8e-3]
ALPHAS_L8 = [3e-3, 5e-3, 6e-3]

print('Setup done')

Setup done


In [2]:
def get_patch_indices(acts):
    N = acts.shape[0] // 197
    img_starts = torch.arange(N) * 197
    return (img_starts.unsqueeze(1) + torch.arange(1, 197).unsqueeze(0)).reshape(-1)

# Checkpoint stats are small — safe to load both upfront
ckpt4 = torch.load('../checkpoints/sae_layer4.pt', weights_only=False)
mean4, rms4 = ckpt4['acts_mean'], ckpt4['acts_rms']
del ckpt4

ckpt8 = torch.load('../checkpoints/sae_layer8.pt', weights_only=False)
mean8, rms8 = ckpt8['acts_mean'], ckpt8['acts_rms']
del ckpt8

print('Loaded normalization stats from checkpoints')

Loaded normalization stats from checkpoints


In [3]:
def resample_dead(sae, acts, patch_indices, mean, rms, opt, dead_mask):
    n_dead = dead_mask.sum().item()
    if n_dead == 0:
        return 0
    pool_pi = patch_indices[torch.randperm(len(patch_indices))[:4096]]
    x_pool  = (acts[pool_pi].float() - mean) / rms.item()
    with torch.no_grad():
        x = x_pool.to(DEVICE)
        _, x_hat = sae(x)
        errors = ((x - x_hat) ** 2).sum(dim=-1)
        chosen = torch.multinomial(errors / (errors.sum() + 1e-8), n_dead, replacement=True)
        dirs = F.normalize(x[chosen], dim=-1)
        sae.W_e.data[dead_mask] = dirs * 0.2
        sae.b_e.data[dead_mask] = 0.0
        sae.W_d.data[:, dead_mask] = dirs.T
        for p in [sae.W_e, sae.b_e, sae.W_d]:
            key = 'exp_avg'
            if p in opt.state and key in opt.state[p]:
                if p is sae.W_d:
                    opt.state[p][key][:, dead_mask] = 0
                    opt.state[p]['exp_avg_sq'][:, dead_mask] = 0
                else:
                    opt.state[p][key][dead_mask] = 0
                    opt.state[p]['exp_avg_sq'][dead_mask] = 0
    return n_dead


def mean_encoder_sim(sae):
    """Mean pairwise cosine similarity of encoder weight rows (lower = more diverse)."""
    W = F.normalize(sae.W_e.data, dim=1)  # (D_HIDDEN, D_INPUT)
    sim = W @ W.T
    sim.fill_diagonal_(0.0)
    return sim.mean().item()


def run_pilot(acts, patch_indices, mean, rms, alpha, label):
    mean_dev = mean.to(DEVICE)
    rms_val  = rms.item()
    n_patch  = len(patch_indices)

    sae = SAE(D_INPUT, D_HIDDEN, alpha=alpha).to(DEVICE)
    opt = torch.optim.Adam(sae.parameters(), lr=LR)

    perm, ptr = torch.randperm(n_patch), 0

    for step in range(PILOT_STEPS):
        if ptr + BATCH > n_patch:
            perm, ptr = torch.randperm(n_patch), 0
        batch_pi = patch_indices[perm[ptr:ptr + BATCH]]; ptr += BATCH
        x = (acts[batch_pi].to(DEVICE).float() - mean_dev) / rms_val

        f, x_hat = sae(x)
        alpha_eff = alpha * min(1.0, step / PILOT_WARMUP)
        total, mse, l1 = sae.loss(x, f, x_hat, alpha=alpha_eff)
        opt.zero_grad(); total.backward(); opt.step()
        sae.normalize_decoder()

        if step == PILOT_RESAMPLE_STEP:
            eval_pi = patch_indices[torch.randperm(n_patch)[:50_000]]
            ever_fired = torch.zeros(D_HIDDEN, dtype=torch.bool)
            sae.eval()
            with torch.no_grad():
                for i in range(0, len(eval_pi), BATCH):
                    xb = (acts[eval_pi[i:i+BATCH]].to(DEVICE).float() - mean_dev) / rms_val
                    ever_fired |= (sae.encode(xb) > 0).any(dim=0).cpu()
            sae.train()
            n_resampled = resample_dead(sae, acts, patch_indices, mean, rms, opt, ~ever_fired)
            print(f'  step {step}: resampled {n_resampled} dead neurons')

        if step % LOG_EVERY == 0 or step == PILOT_STEPS - 1:
            with torch.no_grad():
                l0   = (f > 0).float().sum(dim=1).mean().item()
                dead = ((f > 0).float().sum(dim=0) == 0).float().mean().item() * 100
            print(f'[{label}] step={step:5d} | mse={mse.item():.4f} | '
                  f'l0={l0:.1f} | dead={dead:.1f}% | alpha_eff={alpha_eff:.2e}')

    sim = mean_encoder_sim(sae)
    # Random sample for final stats — avoids bias from first-N images being one class
    eval_idx = patch_indices[torch.randperm(n_patch)[:BATCH]]
    with torch.no_grad():
        xb = (acts[eval_idx].to(DEVICE).float() - mean_dev) / rms_val
        f_final, x_hat_final = sae(xb)
        l0_final   = (f_final > 0).float().sum(dim=1).mean().item()
        dead_final = ((f_final > 0).float().sum(dim=0) == 0).float().mean().item() * 100
        _, mse_final, _ = sae.loss(xb, f_final, x_hat_final)

    print(f'\n→ [{label}] alpha={alpha:.1e} | '
          f'mse={mse_final.item():.4f} | l0={l0_final:.1f} | '
          f'dead={dead_final:.1f}% | mean_sim={sim:.4f}\n')
    return {'alpha': alpha, 'mse': mse_final.item(), 'l0': l0_final,
            'dead': dead_final, 'mean_sim': sim}

In [4]:
print('=' * 60)
print('LAYER 4 ALPHA SWEEP')
print('=' * 60)
acts4 = torch.load('../data/layer4_activations.pt', weights_only=False)
pi4 = get_patch_indices(acts4)

results_l4 = []
for alpha in ALPHAS_L4:
    r = run_pilot(acts4, pi4, mean4, rms4, alpha, f'L4 a={alpha:.0e}')
    results_l4.append(r)

del acts4; gc.collect()

print('\nLayer 4 summary:')
print(f'{"alpha":>10} {"MSE":>8} {"L0":>8} {"dead%":>8} {"mean_sim":>10}')
print('-' * 50)
for r in results_l4:
    flag = '← target' if r['mean_sim'] < 0.20 and r['dead'] < 2 else ''
    print(f"{r['alpha']:>10.1e} {r['mse']:>8.4f} {r['l0']:>8.1f} "
          f"{r['dead']:>8.1f} {r['mean_sim']:>10.4f}  {flag}")

LAYER 4 ALPHA SWEEP
[L4 a=4e-03] step=    0 | mse=1.1857 | l0=1535.5 | dead=0.0% | alpha_eff=0.00e+00
[L4 a=4e-03] step= 2000 | mse=0.1113 | l0=955.4 | dead=0.0% | alpha_eff=8.00e-04
[L4 a=4e-03] step= 4000 | mse=0.1834 | l0=406.7 | dead=0.0% | alpha_eff=1.60e-03
[L4 a=4e-03] step= 6000 | mse=0.2323 | l0=230.5 | dead=0.0% | alpha_eff=2.40e-03
[L4 a=4e-03] step= 8000 | mse=0.2724 | l0=157.1 | dead=0.0% | alpha_eff=3.20e-03
[L4 a=4e-03] step=10000 | mse=0.3101 | l0=118.1 | dead=0.0% | alpha_eff=4.00e-03
[L4 a=4e-03] step=12000 | mse=0.3091 | l0=109.9 | dead=0.0% | alpha_eff=4.00e-03
[L4 a=4e-03] step=14000 | mse=0.3078 | l0=106.1 | dead=0.0% | alpha_eff=4.00e-03
[L4 a=4e-03] step=16000 | mse=0.3056 | l0=104.2 | dead=0.0% | alpha_eff=4.00e-03
[L4 a=4e-03] step=18000 | mse=0.3051 | l0=102.9 | dead=0.0% | alpha_eff=4.00e-03
  step 20000: resampled 0 dead neurons
[L4 a=4e-03] step=20000 | mse=0.3051 | l0=100.9 | dead=0.0% | alpha_eff=4.00e-03
[L4 a=4e-03] step=22000 | mse=0.3088 | l0=101.6 |

In [5]:
print('=' * 60)
print('LAYER 8 ALPHA SWEEP')
print('=' * 60)
acts8 = torch.load('../data/layer8_activations.pt', weights_only=False)
pi8 = get_patch_indices(acts8)

results_l8 = []
for alpha in ALPHAS_L8:
    r = run_pilot(acts8, pi8, mean8, rms8, alpha, f'L8 a={alpha:.0e}')
    results_l8.append(r)

del acts8; gc.collect()

print('\nLayer 8 summary:')
print(f'{"alpha":>10} {"MSE":>8} {"L0":>8} {"dead%":>8} {"mean_sim":>10}')
print('-' * 50)
for r in results_l8:
    flag = '← target' if r['mean_sim'] < 0.20 and r['dead'] < 2 else ''
    print(f"{r['alpha']:>10.1e} {r['mse']:>8.4f} {r['l0']:>8.1f} "
          f"{r['dead']:>8.1f} {r['mean_sim']:>10.4f}  {flag}")

LAYER 8 ALPHA SWEEP
[L8 a=3e-03] step=    0 | mse=1.1842 | l0=1537.6 | dead=0.0% | alpha_eff=0.00e+00
[L8 a=3e-03] step= 2000 | mse=0.1013 | l0=1277.7 | dead=0.0% | alpha_eff=6.00e-04
[L8 a=3e-03] step= 4000 | mse=0.1910 | l0=646.3 | dead=0.0% | alpha_eff=1.20e-03
[L8 a=3e-03] step= 6000 | mse=0.2584 | l0=351.6 | dead=0.0% | alpha_eff=1.80e-03
[L8 a=3e-03] step= 8000 | mse=0.3079 | l0=208.9 | dead=0.0% | alpha_eff=2.40e-03
[L8 a=3e-03] step=10000 | mse=0.3439 | l0=138.2 | dead=0.0% | alpha_eff=3.00e-03
[L8 a=3e-03] step=12000 | mse=0.3417 | l0=124.9 | dead=0.0% | alpha_eff=3.00e-03
[L8 a=3e-03] step=14000 | mse=0.3414 | l0=117.9 | dead=0.0% | alpha_eff=3.00e-03
[L8 a=3e-03] step=16000 | mse=0.3360 | l0=114.1 | dead=0.0% | alpha_eff=3.00e-03
[L8 a=3e-03] step=18000 | mse=0.3356 | l0=111.8 | dead=0.0% | alpha_eff=3.00e-03
  step 20000: resampled 0 dead neurons
[L8 a=3e-03] step=20000 | mse=0.3371 | l0=110.1 | dead=0.0% | alpha_eff=3.00e-03
[L8 a=3e-03] step=22000 | mse=0.3298 | l0=108.2 

## How to choose alpha

Pick the **largest alpha where mean_sim < 0.20 and dead < 2%**.
Larger alpha = sparser (better), smaller alpha = lower redundancy (better diversity).
If no candidate hits mean_sim < 0.20, pick the one with the lowest mean_sim and dead ≈ 0.

Then update `ALPHA_L4` / `ALPHA_L8` in `02_sae_training.ipynb` and run full 100k training.